In [1]:
import random
import re
from collections import Counter, defaultdict
import math
import pandas as pd


In [2]:

with open("hindi_sentences.txt", "r", encoding="utf-8") as f:
    corpus = [line.strip().split() for line in f if line.strip()]

# Add <s>, </s> markers for sentence boundaries
sentences = [["<s>"] + sent + ["</s>"] for sent in corpus]

print("Total sentences:", len(sentences))
print("Example sentence:", sentences[0][:15])

Total sentences: 163148
Example sentence: ['<s>', 'लोगों', 'को', 'बिलों', 'संबंधी', 'सुविधा', 'देना', 'ही', 'उनका', 'काम', '</s>']


In [4]:
random.shuffle(sentences)

validation_set = sentences[:1000]
test_set = sentences[1000:2000]
train_set = sentences[2000:10000]

print(f"Train: {len(train_set)}, Validation: {len(validation_set)}, Test: {len(test_set)}")


Train: 8000, Validation: 1000, Test: 1000


In [5]:
def build_ngram_counts(corpus, n):
    ngram_counts = Counter()
    context_counts = Counter()
    for sent in corpus:
        for i in range(len(sent) - n + 1):
            ngram = tuple(sent[i:i+n])
            context = tuple(sent[i:i+n-1]) if n > 1 else ()
            ngram_counts[ngram] += 1
            context_counts[context] += 1
    return ngram_counts, context_counts

def n_gram(sentence, n):
   
    if len(sentence) < n:
        return []
    return [tuple(sentence[i:i+n]) for i in range(len(sentence)-n+1)]


unigram_counts, _ = build_ngram_counts(train_set, 1)
bigram_counts, bigram_contexts = build_ngram_counts(train_set, 2)
trigram_counts, trigram_contexts = build_ngram_counts(train_set, 3)
quadgram_counts, quad_contexts = build_ngram_counts(train_set, 4)

In [7]:
import pandas as pd

def good_turing_discount(counts, max_freq=100):
    Nc = Counter(counts.values())
    freq_table = []
    adjusted_counts = {}
    
    for c in range(max_freq+1):
        Nc_val = Nc.get(c, 0)
        Nc1_val = Nc.get(c+1, 0)
        if Nc_val > 0:
            c_star = (c+1) * Nc1_val / Nc_val if Nc1_val > 0 else c
            freq_table.append([c, Nc_val, c_star])
        else:
            freq_table.append([c, 0, 0])
    
    # Apply discount
    for ngram, c in counts.items():
        Nc_val = Nc[c]
        Nc1_val = Nc.get(c+1, 0)
        if Nc1_val > 0:
            c_star = (c+1) * Nc1_val / Nc_val
            adjusted_counts[ngram] = c_star
        else:
            adjusted_counts[ngram] = c
    
    df = pd.DataFrame(freq_table, columns=["C (MLE)", "Nc", "C*"])
    return adjusted_counts, df


In [8]:
# Cell 1: build counts and context summaries
from collections import Counter, defaultdict

# Build n-gram counts (uses your n_gram function)
uni_counts, uni_total = build_ngram_counts(train_set, 1)
bi_counts, bi_total = build_ngram_counts(train_set, 2)
tri_counts, tri_total = build_ngram_counts(train_set, 3)
quad_counts, quad_total = build_ngram_counts(train_set, 4)

counts_by_n = {1: uni_counts, 2: bi_counts, 3: tri_counts, 4: quad_counts}

def context_summaries(counts):
    ctx_sum = defaultdict(int)
    ctx_seen = defaultdict(set)
    for ng, c in counts.items():
        if len(ng) == 1:
            ctx = ()
            ctx_sum[ctx] += c
            ctx_seen[ctx].add(ng[0])
        else:
            ctx = ng[:-1]
            ctx_sum[ctx] += c
            ctx_seen[ctx].add(ng[-1])
    return ctx_sum, ctx_seen

uni_ctx_sum, uni_ctx_seen = context_summaries(uni_counts)
bi_ctx_sum, bi_ctx_seen   = context_summaries(bi_counts)
tri_ctx_sum, tri_ctx_seen = context_summaries(tri_counts)
quad_ctx_sum, quad_ctx_seen = context_summaries(quad_counts)

context_sum_by_n = {1: uni_ctx_sum, 2: bi_ctx_sum, 3: tri_ctx_sum, 4: quad_ctx_sum}
context_seen_by_n = {1: uni_ctx_seen, 2: bi_ctx_seen, 3: tri_ctx_seen, 4: quad_ctx_seen}

# vocabulary (exclude <s> maybe, we will keep full vocab)
vocab = set([w for (w,),_ in uni_counts.items()])
vocab_list = list(vocab)
print("Vocab size:", len(vocab))


Vocab size: 22304


In [9]:
# Cell 2: Good-Turing adjusted counts (we use these for Katz backoff c*)
# If you already ran good_turing_discount, skip/overwrite as appropriate.
uni_gt, uni_freq_table = good_turing_discount(uni_counts)
bi_gt, bi_freq_table   = good_turing_discount(bi_counts)
tri_gt, tri_freq_table = good_turing_discount(tri_counts)
quad_gt, quad_freq_table = good_turing_discount(quad_counts)

adj_by_n = {1: uni_gt, 2: bi_gt, 3: tri_gt, 4: quad_gt}

# Precompute context sums and seen tokens for adjusted counts
from collections import defaultdict
adj_ctx_sum = {n: defaultdict(float) for n in range(1,5)}
adj_ctx_seen = {n: defaultdict(set) for n in range(1,5)}

for n, adj in adj_by_n.items():
    for ng, cstar in adj.items():
        if n == 1:
            ctx = ()
            adj_ctx_sum[n][ctx] += cstar
            adj_ctx_seen[n][ctx].add(ng[0])
        else:
            ctx = ng[:-1]
            adj_ctx_sum[n][ctx] += cstar
            adj_ctx_seen[n][ctx].add(ng[-1])

total_cstar_unigram = sum(adj_by_n[1].values())


In [10]:
# Cell 3: Katz backoff implementation using Good-Turing c*
from functools import lru_cache
import math

# Short names
adj = adj_by_n
adj_ctxS = adj_ctx_sum
adj_seen = adj_ctx_seen

@lru_cache(maxsize=None)
def katz_prob(ngram):
    """
    Recursive Katz backoff probability for an ngram (tuple).
    Uses adjusted counts (c*) for seen ngrams and computes backoff weights alpha.
    """
    n = len(ngram)
    # Unigram base case
    if n == 1:
        if ngram in adj[1] and total_cstar_unigram > 0:
            return adj[1][ngram] / total_cstar_unigram
        else:
            # small probability for unseen unigram (practical fallback)
            return 1e-12

    # If seen at this order, use normalized c*
    if ngram in adj[n]:
        ctx = ngram[:-1]
        denom = adj_ctxS[n].get(ctx, 0.0)
        if denom > 0:
            return adj[n][ngram] / denom
        else:
            return 1e-12

    # Otherwise backoff
    ctx = ngram[:-1]
    seen_set = adj_seen[n].get(ctx, set())
    denom_ctx = adj_ctxS[n].get(ctx, 0.0)

    # sum of seen probs at this level (using c* normalization)
    sum_seen_probs = 0.0
    if denom_ctx > 0:
        for w in seen_set:
            sum_seen_probs += adj[n].get(ctx + (w,), 0.0) / denom_ctx

    # sum of lower-order probs for the same seen tokens
    lower_sum = 0.0
    for w in seen_set:
        lower_ngram = ctx[1:] + (w,) if len(ctx) >= 1 else (w,)
        lower_sum += katz_prob(lower_ngram)

    denom = 1.0 - lower_sum
    alpha = (1.0 - sum_seen_probs) / denom if denom > 0 else 1.0

    # target lower-order ngram for the specific unseen continuation
    lower_target = katz_prob(ngram[1:])
    return alpha * lower_target

# Sentence probability using Katz quadrigram
def sentence_prob_katz(sentence):
    probs = 1.0
    for qg in n_gram(sentence, 4):
        p = katz_prob(qg)
        probs *= p if p > 0 else 1e-12
    return probs

# Evaluate on validation and test sets (returns list of probs)
val_probs_katz = [sentence_prob_katz(s) for s in validation_set]
test_probs_katz = [sentence_prob_katz(s) for s in test_set]

print("Example Katz prob (validation):", val_probs_katz[0])


Example Katz prob (validation): 0.0


In [11]:
# Cell 4: Interpolated Kneser-Ney (quad) implementation
from functools import lru_cache

D = 0.75  # discount; can be tuned

# Precompute useful things for Kneser-Ney
# 1) continuation counts for unigram (number of distinct left contexts that precede w)
continuation_precede = defaultdict(set)
for (w1, w2), c in bi_counts.items():
    continuation_precede[w2].add(w1)
total_bigram_types = len(bi_counts)
continuation_count_unigram = {w: len(continuation_precede[w]) for w in continuation_precede}

@lru_cache(maxsize=None)
def kn_prob(ngram):
    """
    Interpolated Kneser-Ney probability for any order ngram.
    For n==1 base: use continuation probability P_cont(w) = (# contexts that precede w) / total_bigram_types
    For n>1:
      P = max(c(h,w)-D,0)/c(h) + lambda(h) * P_lower(w | h_suffix)
      lambda(h) = D * |{w': c(h,w')>0}| / c(h)
    """
    n = len(ngram)
    if n == 1:
        w = ngram[0]
        return continuation_count_unigram.get(w, 0) / total_bigram_types if total_bigram_types > 0 else 0.0

    counts = counts_by_n[n]
    ctx = ngram[:-1]
    c_hw = counts.get(ngram, 0)
    c_h = context_sum_by_n[n].get(ctx, 0)
    if c_h == 0:
        # no context observed -> backoff to lower order
        return kn_prob(ngram[1:])

    first_term = max(c_hw - D, 0) / c_h
    num_types = len(context_seen_by_n[n].get(ctx, []))
    lambda_h = (D * num_types) / c_h if c_h > 0 else 0.0
    lower_prob = kn_prob(ngram[1:])
    return first_term + lambda_h * lower_prob

# Compute KN sentence probabilities on validation/test
def sentence_prob_kn(sentence):
    probs = 1.0
    for qg in n_gram(sentence, 4):
        p = kn_prob(qg)
        probs *= p if p > 0 else 1e-12
    return probs

val_probs_kn = [sentence_prob_kn(s) for s in validation_set]
test_probs_kn = [sentence_prob_kn(s) for s in test_set]

print("Example Kneser-Ney prob (validation):", val_probs_kn[0])


Example Kneser-Ney prob (validation): 2.7618220885803287e-303


In [12]:
# Cell 5: Greedy generator using MLE/backoff
import math

def ml_next_tokens(context, order, counts_by_n, context_sum_by_n):
    """
    return dict(w -> P_ml(w | context)) using MLE for given order (with simple backoff).
    context is a tuple of length order-1
    """
    n = order
    counts = counts_by_n.get(n)
    ctx_sum = context_sum_by_n.get(n)
    ctx = tuple(context)
    probs = {}
    if ctx_sum.get(ctx, 0) > 0:
        # tokens seen for this context
        for w in context_seen_by_n[n].get(ctx, []):
            probs[w] = counts.get(ctx + (w,), 0) / ctx_sum[ctx]
        return probs
    else:
        # backoff to lower order
        if n == 1:
            # unigram fallback
            total = sum(counts_by_n[1].values())
            for (w,), c in counts_by_n[1].items():
                probs[w] = c / total if total > 0 else 0.0
            return probs
        else:
            # recurse with shorter context
            return ml_next_tokens(context[1:], order-1, counts_by_n, context_sum_by_n)

def greedy_generate(order, counts_by_n, context_sum_by_n, max_len=30):
    """
    Generate one sentence greedily for given order (1..4).
    """
    history = ["<s>"] * (order - 1)
    sent = list(history)
    for _ in range(max_len):
        probs = ml_next_tokens(tuple(history), order, counts_by_n, context_sum_by_n)
        if not probs:
            # empty -> pick most frequent unigram
            next_tok = max(counts_by_n[1].items(), key=lambda x: x[1])[0][0]
        else:
            # choose token with highest probability
            next_tok = max(probs.items(), key=lambda x: x[1])[0]
        sent.append(next_tok)
        if next_tok == "</s>":
            break
        # shift history
        if order > 1:
            history = (history + [next_tok])[1:]
    return sent

# Generate 100 sentences per n-gram model (greedy)
generated_greedy = {1: [], 2: [], 3: [], 4: []}
for n in (1,2,3,4):
    print(f"Generating greedy sentences for order {n} ...")
    for i in range(100):
        s = greedy_generate(n, counts_by_n, context_sum_by_n)
        generated_greedy[n].append(s)
    print(f"Done order {n}: first example:", generated_greedy[n][0])


Generating greedy sentences for order 1 ...
Done order 1: first example: ['<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>', '<s>']
Generating greedy sentences for order 2 ...
Done order 2: first example: ['<s>', 'इस', 'दौरान', 'एक', 'बार', 'फिर', 'से', 'अधिक', 'लोगों', 'को', 'लेकर', 'चल', 'रहे', 'हैं।', '</s>']
Generating greedy sentences for order 3 ...
Done order 3: first example: ['<s>', '<s>', 'इस', 'दौरान', 'एक', 'दूरी', 'कायम', 'रखने', 'के', 'लिए', 'एक', 'साथ', '100', 'यात्रियों', 'के', 'बैठने', 'की', 'व्यवस्था', 'की', 'गई', 'है।', '</s>']
Generating greedy sentences for order 4 ...
Done order 4: first example: ['<s>', '<s>', '<s>', 'इस', 'दौरान', 'उनके', 'फॉरेक्स', 'कार्ड', 'से', '428.56', 'यूरो', 'और', '0.49', 'यूएस', 'डॉलर', 'का', 'ट्रांजेक्शन', 'किया', 'गया', 'था।', '</s>']


In [13]:
# Cell 6: Beam search generation
import heapq
import math

def ml_prob_for_token(context, token, order, counts_by_n, context_sum_by_n):
    # returns MLE probability P(token | context) with simple backoff
    ctx = tuple(context)
    n = order
    if context_sum_by_n[n].get(ctx, 0) > 0:
        return counts_by_n[n].get(ctx + (token,), 0) / context_sum_by_n[n][ctx]
    else:
        if n == 1:
            total = sum(counts_by_n[1].values())
            return counts_by_n[1].get((token,), 0) / total if total > 0 else 0.0
        else:
            return ml_prob_for_token(ctx[1:], token, order-1, counts_by_n, context_sum_by_n)

def beam_search_generate(order, beam_size=20, max_len=30, top_k_candidates=50):
    # initial beam: (logprob, history_list)
    init_history = ["<s>"] * (order - 1)
    beam = [(0.0, init_history[:])]  # logprob 0
    completed = []

    for step in range(max_len):
        candidates = []
        for logp, history in beam:
            # If already ended, keep it in completed
            if len(history) > 0 and history[-1] == "</s>":
                candidates.append((logp, history))
                continue

            # Candidate tokens: tokens seen after the history in this order
            ctx = tuple(history[-(order-1):]) if order > 1 else tuple()
            tokens_set = context_seen_by_n[order].get(ctx, None)
            if not tokens_set or len(tokens_set) == 0:
                # backoff: use frequent unigrams as candidate pool
                # take top_k_candidates unigrams by count
                tokens_list = [w for (w,),_ in uni_counts.most_common(top_k_candidates)]
            else:
                # to keep expansion manageable, take top_k by conditional prob
                token_probs = []
                for w in tokens_set:
                    p = ml_prob_for_token(ctx, w, order, counts_by_n, context_sum_by_n)
                    token_probs.append((p, w))
                token_probs.sort(reverse=True)
                tokens_list = [w for p,w in token_probs[:top_k_candidates]]

            # expand beam for each candidate token
            for w in tokens_list:
                p = ml_prob_for_token(ctx, w, order, counts_by_n, context_sum_by_n)
                if p <= 0:
                    continue
                new_logp = logp + math.log(p)
                new_history = history[:] + [w]
                candidates.append((new_logp, new_history))

        # keep top beam_size candidates by logprob
        # use nlargest
        if not candidates:
            break
        beam = heapq.nlargest(beam_size, candidates, key=lambda x: x[0])

        # stop if all beams ended with </s>
        if all(h[-1] == "</s>" for _, h in beam):
            break

    # collect completed sequences from beam (prefer ones ending with </s>)
    completed = [h for _, h in beam if h[-1] == "</s>"]
    # if none completed, return top beam sequences (trim to sentences)
    if not completed:
        completed = [h for _, h in beam]

    # return sequence list (remove leading <s> tokens)
    return [seq for seq in completed]

# Generate 100 sentences per n-gram model (beam search)
generated_beam = {1: [], 2: [], 3: [], 4: []}
for n in (1,2,3,4):
    print(f"Beam generating for order {n} ...")
    gen = []
    # beam returns up to beam_size sequences per call, so iterate until 100 collected
    i = 0
    while len(gen) < 100:
        out_seqs = beam_search_generate(n, beam_size=20, max_len=30, top_k_candidates=50)
        # beam returns many candidates; add until we have 100 unique or repeated allowed
        for s in out_seqs:
            if len(gen) >= 100:
                break
            gen.append(s)
        i += 1
        if i > 200:  # safety to prevent infinite loop on degenerate data
            break
    generated_beam[n] = gen[:100]
    print(f"Done order {n}: example:", generated_beam[n][0])


Beam generating for order 1 ...
Done order 1: example: ['</s>']
Beam generating for order 2 ...
Done order 2: example: ['<s>', 'नई', 'दिल्ली।', '</s>']
Beam generating for order 3 ...
Done order 3: example: ['<s>', '<s>', 'इस', 'मौके', 'पर', 'ही', 'मौत', 'हो', 'गई।', '</s>']
Beam generating for order 4 ...
Done order 4: example: ['<s>', '<s>', '<s>', 'यह', 'भी', 'पढ़ें:', 'इस', 'साल', '1.5', 'लाख', 'स्कूली', 'बच्चियों', 'को', 'स्कॉलरशिप', 'देगा', 'अल्पसंख्यक', 'मंत्रालय', '</s>']


In [14]:
# Cell 7: show examples and save to files
for n in (1,2,3,4):
    print(f"\nOrder {n} — greedy example:")
    print(" ".join(generated_greedy[n][0]))
    print(f"Order {n} — beam example:")
    print(" ".join(generated_beam[n][0]))

# Save to disk if you want
import json
with open("generated_greedy.json", "w", encoding="utf-8") as f:
    json.dump(generated_greedy, f, ensure_ascii=False, indent=2)
with open("generated_beam.json", "w", encoding="utf-8") as f:
    json.dump(generated_beam, f, ensure_ascii=False, indent=2)
print("Saved generated sentences to generated_greedy.json and generated_beam.json")



Order 1 — greedy example:
<s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s> <s>
Order 1 — beam example:
</s>

Order 2 — greedy example:
<s> इस दौरान एक बार फिर से अधिक लोगों को लेकर चल रहे हैं। </s>
Order 2 — beam example:
<s> नई दिल्ली। </s>

Order 3 — greedy example:
<s> <s> इस दौरान एक दूरी कायम रखने के लिए एक साथ 100 यात्रियों के बैठने की व्यवस्था की गई है। </s>
Order 3 — beam example:
<s> <s> इस मौके पर ही मौत हो गई। </s>

Order 4 — greedy example:
<s> <s> <s> इस दौरान उनके फॉरेक्स कार्ड से 428.56 यूरो और 0.49 यूएस डॉलर का ट्रांजेक्शन किया गया था। </s>
Order 4 — beam example:
<s> <s> <s> यह भी पढ़ें: इस साल 1.5 लाख स्कूली बच्चियों को स्कॉलरशिप देगा अल्पसंख्यक मंत्रालय </s>
Saved generated sentences to generated_greedy.json and generated_beam.json
